In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import time
import csv


import gurobipy as gp
from gurobipy import GRB

def solve_mip(tasks, time_horizon):
    """
    tasks: list of lists, each of which has 3 elements representing:
        (0): Task time required (integer)
        (1): Task priority (integer)
        (2): Task deadline (integer)
    """

    # Initialize model 
    m = gp.Model("smart-calendar")

    # Index sets
    J = range(len(tasks))  # Tasks
    H = range(time_horizon)  # Time slots

    # Binary variables: x[i, j] = 1 if task j is assigned to time i
    x = m.addVars(H, J, vtype=GRB.BINARY, name="x")

    # Integer variables: a[j] stores the latest time slot (argmax i)
    a = m.addVars(J, vtype=GRB.INTEGER, lb=0, ub=time_horizon-1, name="a")

    # Extract task attributes
    time_req = [task[0] for task in tasks]
    priority = [task[1] for task in tasks]
    deadline = [task[2] for task in tasks]

    # Constraints

    # (1) Each time slot can have at most one task
    m.addConstrs((gp.quicksum(x[i, j] for j in J) <= 1 for i in H), name="one_task_per_slot")

    # (2) Each task must be assigned at least 'time_req[j]' times before its deadline
    m.addConstrs((gp.quicksum(x[i, j] for i in range(min(deadline[j], time_horizon))) >= time_req[j] for j in J), name="time_requirement")

    # (3) Ensure a[j] is the largest i where x[i, j] = 1
    m.addConstrs((a[j] >= i * x[i, j] for i in H for j in J), name="argmax_constraint")

    # Set objective function: minimize the sum of a[j] (or another criterion)
    m.setObjective(gp.quicksum(a[j] for j in J), GRB.MINIMIZE)

    # Solve model
    m.optimize()

    # Print results
    for v in m.getVars():
        print('%s %g' % (v.VarName, v.X))

# Example test case
# tasks = [
#     [2, 5, 4],  # Task 1: requires 2 time slots, priority 5, deadline 4
#     [3, 2, 6],  # Task 2: requires 3 time slots, priority 2, deadline 6
# ]
tasks = [
    [4, 1, 10],
    [2, 3, 8],
    [1, 1, 10]
]

time_horizon = 10

solve_mip(tasks, time_horizon)


def solve_mip_old(tasks, time_horizon):
    """
    tasks: list of lists, each of which has 3 elements representing:
        (0): Task time required (integer)
        (1): Task priority (integer)
        (2): Task deadline (integer)
    """

    # Initialize model 
    m = gp.Model("smart-calendar")

    # Initialize variables
    J = range(len(tasks))
    H = range(time_horizon)


    # Add binary variables for all (i, j) pairs
    x = m.addVars(H, J, vtype=GRB.BINARY, name="x")
    a = m.addVars(J, vtype=GRB.INTEGER, name="a", lb=0)

    time_req = list()
    priority = list()
    deadline = list()
    # Extract task attributes
    time_req = [task[0] for task in 22]
    priority = [task[1] for task in tasks]
    deadline = [task[2] for task in tasks]
    
    print(f"time_req: {time_req}")
    # Constraints
    m.addConstrs((gp.quicksum(x[i, j] for j in J) <= 1 for i in H))
    m.addConstrs(((gp.quicksum(x[i, j] for i in range(deadline[j])) >= time_req[j]) for j in J))
    
    # (3) Ensure a[j] is the largest i where x[i, j] = 1
    m.addConstrs((a[j] >= i * x[i, j] for i in H for j in J), name="argmax_constraint")

    # Set objective function: minimize the sum of a[j] (or another criterion)
    m.setObjective(gp.quicksum(a[j] for j in J), GRB.MINIMIZE)

    # Solve model
    m.optimize()
    for v in m.getVars():
        print('%s %g' % (v.VarName, v.X))

    
    # print('Obj: %g' % myProgram.ObjVal)



Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.2.0 23C71)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2548635 - for non-commercial use only - registered to ba___@rice.edu
Optimize a model with 43 rows, 33 columns and 115 nonzeros
Model fingerprint: 0x41444baa
Variable types: 0 continuous, 33 integer (30 binary)
Coefficient statistics:
  Matrix range     [1e+00, 9e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 9e+00]
  RHS range        [1e+00, 4e+00]
Found heuristic solution: objective 24.0000000
Presolve removed 5 rows and 2 columns
Presolve time: 0.00s
Presolved: 38 rows, 31 columns, 106 nonzeros
Variable types: 0 continuous, 31 integer (28 binary)

Root relaxation: objective 3.104614e+00, 40 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | 

In [69]:


result = solve_mip_old(tasks=tasks, time_horizon=15)
print(result)

time_req: [4, 2, 1]
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.2.0 23C71)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2548635 - for non-commercial use only - registered to ba___@rice.edu
Optimize a model with 63 rows, 48 columns and 160 nonzeros
Model fingerprint: 0xb3439e72
Variable types: 0 continuous, 48 integer (45 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]
Found heuristic solution: objective 24.0000000
Presolve removed 25 rows and 17 columns
Presolve time: 0.00s
Presolved: 38 rows, 31 columns, 106 nonzeros
Variable types: 0 continuous, 31 integer (28 binary)

Root relaxation: objective 3.104614e+00, 40 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumb